# Economics Experimental Design & RCT Simulator – Solution Notebook

**Economics Application – Experimental Design, RCTs & Causal Inference Education**

Adapted from the Clinical Trial Randomization Simulator (itself adapted from Rock Paper Scissors).
This tool teaches why proper randomization is the foundation of causal identification in
economic experiments, field RCTs, and A/B tests by letting you simulate different allocation
strategies and observe balance versus selection bias over many replications.

---
## Flowchart of the Desired Outcome

In [1]:
from IPython.display import Image, display
display(Image(filename='economics_rct_flowchart.png', width=850))

## 1. Imports and Experiment Parameters

We define a two-arm economic experiment or RCT: Treatment (intervention) vs Control (status quo).

In [2]:
import random
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# Core experiment settings
N_SUBJECTS = 100          # number of subjects / units to randomize
ARMS = ['Treatment', 'Control']
TARGET_RATIO = 0.5        # target proportion in Treatment arm

print(f'Experiment size: {N_SUBJECTS} subjects')
print(f'Arms: {ARMS}')
print(f'Target allocation to Treatment: {TARGET_RATIO*100:.0f}%')

Experiment size: 100 subjects
Arms: ['Treatment', 'Control']
Target allocation to Treatment: 50%


## 2. Allocation Methods (Primary + Alternates)

### Primary: Simple randomization (independent coin flip for each subject)
### Alternate 1: Permuted-block randomization (guarantees balance within blocks)
### Alternate 2: Biased / deterministic strategies (for teaching selection bias)

In [3]:
def simple_randomization(n_subjects, p_treatment=0.5, seed=None):
    """Independent Bernoulli trial for each subject."""
    if seed is not None:
        random.seed(seed)
    return ['Treatment' if random.random() < p_treatment else 'Control'
            for _ in range(n_subjects)]

def block_randomization(n_subjects, block_size=4, p_treatment=0.5, seed=None):
    """Permuted-block randomization. block_size should be even for 1:1."""
    if seed is not None:
        random.seed(seed)
    assignments = []
    n_treat_per_block = int(block_size * p_treatment)
    while len(assignments) < n_subjects:
        block = (['Treatment'] * n_treat_per_block +
                 ['Control'] * (block_size - n_treat_per_block))
        random.shuffle(block)
        assignments.extend(block)
    return assignments[:n_subjects]

def biased_always_treatment(n_subjects):
    """Deterministic – every subject to Treatment (extreme selection bias)."""
    return ['Treatment'] * n_subjects

def biased_sequential(n_subjects):
    """Alternate T/C/T/C... (predictable, not random)."""
    return ['Treatment' if i % 2 == 0 else 'Control' for i in range(n_subjects)]

# Quick demo
print('Simple (seed=42):', simple_randomization(10, seed=42))
print('Block  (seed=42):', block_randomization(10, block_size=4, seed=42))
print('Always Treatment:', biased_always_treatment(6))
print('Sequential:      ', biased_sequential(6))

Simple (seed=42): ['Control', 'Control', 'Treatment', 'Treatment', 'Control', 'Treatment', 'Treatment', 'Control', 'Control', 'Control']
Block  (seed=42): ['Control', 'Treatment', 'Treatment', 'Control', 'Control', 'Treatment', 'Treatment', 'Control', 'Treatment', 'Control']
Always Treatment: ['Treatment', 'Treatment', 'Treatment', 'Treatment', 'Treatment', 'Treatment']
Sequential:       ['Treatment', 'Control', 'Treatment', 'Control', 'Treatment', 'Control']


## 3. Single-Experiment Summary Function

Given a list of assignments, compute counts, percentage in Treatment, and absolute imbalance.

In [4]:
def summarize_experiment(assignments):
    """Return dict with counts, pct_treatment, and imbalance."""
    counts = Counter(assignments)
    n = len(assignments)
    n_treat = counts.get('Treatment', 0)
    pct = n_treat / n if n else 0
    imbalance = abs(n_treat - (n - n_treat))   # |T - C|
    return {
        'n': n,
        'Treatment': n_treat,
        'Control': counts.get('Control', 0),
        'pct_treatment': pct,
        'imbalance': imbalance
    }

# Demo on the methods
for name, func in [('Simple', lambda: simple_randomization(100, seed=1)),
                   ('Block',  lambda: block_randomization(100, seed=1)),
                   ('Always T', biased_always_treatment),
                   ('Sequential', biased_sequential)]:
    s = summarize_experiment(func(100) if name not in ('Always T', 'Sequential') else func(100))
    print(f"{name:12s} → T={s['Treatment']:3d}  C={s['Control']:3d}  "
          f"pct={s['pct_treatment']*100:5.1f}%  imbalance={s['imbalance']}")

Simple       → T= 52  C= 48  pct= 52.0%  imbalance=4
Block        → T= 50  C= 50  pct= 50.0%  imbalance=0
Always T     → T=100  C=  0  pct=100.0%  imbalance=100
Sequential   → T= 50  C= 50  pct= 50.0%  imbalance=0


## 4. Monte-Carlo Simulation Engine

Run many independent experiments and collect the distribution of the Treatment percentage.
This demonstrates the operating characteristics of each allocation rule.

In [5]:
def run_monte_carlo(method_func, n_subjects=100, n_trials=2000, seed=42):
    """
    method_func: function that takes n_subjects and returns a list of assignments.
    Returns list of pct_treatment from each simulated experiment.
    """
    random.seed(seed)
    pcts = []
    for i in range(n_trials):
        assignments = method_func(n_subjects)
        s = summarize_experiment(assignments)
        pcts.append(s['pct_treatment'])
    return pcts

# Example: simple randomization
pcts_simple = run_monte_carlo(lambda n: simple_randomization(n), n_subjects=100, n_trials=2000)
print(f'Simple randomization – mean % Treatment: {np.mean(pcts_simple)*100:.1f}%')
print(f'                       std:               {np.std(pcts_simple)*100:.1f}%')
print(f'                       min–max:           {min(pcts_simple)*100:.1f}% – {max(pcts_simple)*100:.1f}%')

Simple randomization – mean % Treatment: 50.0%
                       std:               5.0%
                       min–max:           34.0% – 66.0%


## 5. Visualization of Allocation Distributions

Histograms show how tightly the % Treatment clusters around the target.
Fair methods produce a concentrated distribution; biased methods produce spikes far from the target.

In [6]:
N_TRIALS = 2000
N_SUB = 100

pcts = {
    'Simple Random': run_monte_carlo(lambda n: simple_randomization(n), N_SUB, N_TRIALS, seed=10),
    'Block (size=4)': run_monte_carlo(lambda n: block_randomization(n, block_size=4), N_SUB, N_TRIALS, seed=10),
    'Always Treatment': run_monte_carlo(lambda n: biased_always_treatment(n), N_SUB, N_TRIALS, seed=10),
    'Sequential': run_monte_carlo(lambda n: biased_sequential(n), N_SUB, N_TRIALS, seed=10),
}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.flatten()
colors = ['#27AE60', '#2980B9', '#E74C3C', '#F39C12']

for ax, (name, data), color in zip(axes, pcts.items(), colors):
    ax.hist([p*100 for p in data], bins=20, color=color, edgecolor='black', alpha=0.85)
    ax.axvline(50, color='black', linestyle='--', linewidth=1.5, label='Target 50%')
    ax.set_title(name)
    ax.set_xlabel('% assigned to Treatment')
    ax.set_ylabel('Number of simulated experiments')
    ax.set_xlim(0, 100)
    ax.legend(fontsize=8)

plt.suptitle(f'Distribution of Treatment Allocation\n({N_TRIALS} experiments × {N_SUB} subjects)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('economics_rct_simulation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → economics_rct_simulation.png')

Saved → economics_rct_simulation.png


## 6. Strategy Comparison Table

In [7]:
print(f'{"Method":20s}  {"Mean %T":>8s}  {"Std":>6s}  {"Min":>6s}  {"Max":>6s}  {"P(|imb|>10)":>12s}')
print('-'*70)
for name, data in pcts.items():
    arr = np.array(data) * 100
    imb = np.abs(arr - 50)
    p_bad = np.mean(imb > 10) * 100
    print(f'{name:20s}  {arr.mean():8.1f}  {arr.std():6.1f}  {arr.min():6.1f}  {arr.max():6.1f}  {p_bad:10.1f}%')

Method                 Mean %T     Std     Min     Max  P(|imb|>10)
----------------------------------------------------------------------
Simple Random             50.1     5.1    34.0    66.0        5.1%
Block (size=4)            50.0     0.0    50.0    50.0        0.0%
Always Treatment         100.0     0.0   100.0   100.0      100.0%
Sequential                50.0     0.0    50.0    50.0        0.0%


## 7. More Practice Exercises

### Practice A – 2:1 Allocation (common in some development RCTs)
Target 2/3 of subjects to Treatment. Verify both simple and block methods centre near 66.7%.

In [8]:
pcts_2to1 = run_monte_carlo(lambda n: simple_randomization(n, p_treatment=2/3), n_subjects=90, n_trials=1000, seed=7)
print(f'2:1 Simple – mean % Treatment: {np.mean(pcts_2to1)*100:.1f}%  (target ≈ 66.7%)')

pcts_block_2to1 = run_monte_carlo(lambda n: block_randomization(n, block_size=6, p_treatment=2/3),
                                  n_subjects=90, n_trials=1000, seed=7)
print(f'2:1 Block  – mean % Treatment: {np.mean(pcts_block_2to1)*100:.1f}%  (target ≈ 66.7%)')

2:1 Simple – mean % Treatment: 66.7%  (target ≈ 66.7%)
2:1 Block  – mean % Treatment: 66.7%  (target ≈ 66.7%)


### Practice B – Chance Imbalance and Sample Size
Even fair simple randomization can produce large imbalances by chance when N is small.
Explore how P(|%T − 50| > 10) changes with sample size — a key design consideration for economic experiments.

In [9]:
print('Probability of |%T - 50| > 10 under simple randomization:')
for n in [20, 50, 100, 200, 500]:
    pcts = run_monte_carlo(lambda n_: simple_randomization(n_), n_subjects=n, n_trials=3000, seed=99)
    p = np.mean(np.abs(np.array(pcts)*100 - 50) > 10) * 100
    print(f'  N = {n:3d}  →  {p:5.1f}%')

Probability of |%T - 50| > 10 under simple randomization:
  N =  20  →   50.3%
  N =  50  →   20.2%
  N = 100  →    5.1%
  N = 200  →    0.5%
  N = 500  →    0.0%


## 8. Tunable Simulation Section

**Modify the parameters below and re-run** to explore different experimental designs.

In [10]:
# ========== TUNABLE PARAMETERS ==========
N_SUBJECTS   = 80
N_TRIALS     = 1500
METHOD       = 'simple'   # 'simple' | 'block' | 'always_t' | 'sequential'
BLOCK_SIZE   = 4
P_TREATMENT  = 0.5
SEED         = 123
# ========================================

def get_method(method):
    if method == 'simple':
        return lambda n: simple_randomization(n, p_treatment=P_TREATMENT)
    elif method == 'block':
        return lambda n: block_randomization(n, block_size=BLOCK_SIZE, p_treatment=P_TREATMENT)
    elif method == 'always_t':
        return biased_always_treatment
    elif method == 'sequential':
        return biased_sequential
    else:
        raise ValueError('Unknown method')

pcts_tune = run_monte_carlo(get_method(METHOD), n_subjects=N_SUBJECTS, n_trials=N_TRIALS, seed=SEED)
arr = np.array(pcts_tune) * 100
print(f'Method={METHOD} | N={N_SUBJECTS} | trials={N_TRIALS}')
print(f'Mean % Treatment : {arr.mean():.1f}%')
print(f'Std              : {arr.std():.1f}%')
print(f'Range            : {arr.min():.1f}% – {arr.max():.1f}%')
print(f'P(|dev| > 10)    : {np.mean(np.abs(arr-50)>10)*100:.1f}%')

Method=simple | N=80 | trials=1500
Mean % Treatment : 50.0%
Std              : 5.6%
Range            : 32.5% – 67.5%
P(|dev| > 10)    : 8.7%


## 9. Key Educational Insights for Economists

1. Changing a fair random process into a deterministic rule (e.g., always assigning the intervention)
   introduces selection bias and destroys the ability to make causal claims — the central reason
   experimental economists and RCT practitioners insist on proper randomization.

2. Even fair simple randomization can produce noticeable imbalance by chance when samples are small.
   This is why many economic experiments use block or stratified randomization, and why we still
   check covariate balance after randomization.

3. The Monte-Carlo approach used here is the same technique applied in power calculations and
   pre-analysis plans for real economic experiments and RCTs.

## Key Takeaways

- Proper randomization (simple or blocked) keeps expected allocation on target and limits chance imbalance.
- Deterministic or biased strategies create systematic imbalance — the educational analogue of selection bias that invalidates causal inference.
- Sample size matters: larger N makes large chance imbalances rare under simple randomization.
- Monte-Carlo simulation is a practical way to evaluate an experimental design before running a real study.
- The same programming patterns used in simple games and clinical-trial teaching tools transfer directly to rigorous training in experimental economics and RCT design.